In [2]:
import tkinter as tk
from tkinter import messagebox
import copy

# ---------------------------- AVL 树节点 ----------------------------
class AVLNode:
    def __init__(self, key):
        self.key = key
        self.left = None
        self.right = None
        self.height = 1   # 叶子节点高度为1

# ---------------------------- AVL 树 ----------------------------
class AVLTree:
    def __init__(self, root=None):
        self.root = root
        self.last_rotation_type = None
        self.last_pivot_key = None

    # 深拷贝整棵树（用于保存上一步状态）
    @staticmethod
    def copy_tree(node):
        if not node:
            return None
        new_node = AVLNode(node.key)
        new_node.height = node.height
        new_node.left = AVLTree.copy_tree(node.left)
        new_node.right = AVLTree.copy_tree(node.right)
        return new_node

    def _height(self, node):
        return node.height if node else 0

    def _update_height(self, node):
        if node:
            node.height = 1 + max(self._height(node.left), self._height(node.right))

    def _balance_factor(self, node):
        return self._height(node.left) - self._height(node.right) if node else 0

    # 右旋 (LL 型)
    def _rotate_right(self, y):
        x = y.left
        T2 = x.right
        x.right = y
        y.left = T2
        self._update_height(y)
        self._update_height(x)
        return x

    # 左旋 (RR 型)
    def _rotate_left(self, x):
        y = x.right
        T2 = y.left
        y.left = x
        x.right = T2
        self._update_height(x)
        self._update_height(y)
        return y

    def insert(self, key):
        self.last_rotation_type = None
        self.last_pivot_key = None
        self.root = self._insert_rec(self.root, key)
        return self.last_rotation_type, self.last_pivot_key

    def _insert_rec(self, node, key):
        if not node:
            return AVLNode(key)

        if key < node.key:
            node.left = self._insert_rec(node.left, key)
        elif key > node.key:
            node.right = self._insert_rec(node.right, key)
        else:
            return node

        self._update_height(node)
        balance = self._balance_factor(node)

        # LL
        if balance > 1 and key < node.left.key:
            self.last_rotation_type = "LL"
            self.last_pivot_key = node.left.key
            return self._rotate_right(node)

        # RR
        if balance < -1 and key > node.right.key:
            self.last_rotation_type = "RR"
            self.last_pivot_key = node.right.key
            return self._rotate_left(node)

        # LR
        if balance > 1 and key > node.left.key:
            self.last_rotation_type = "LR"
            pivot = node.left.right
            self.last_pivot_key = pivot.key if pivot else None
            node.left = self._rotate_left(node.left)
            return self._rotate_right(node)

        # RL
        if balance < -1 and key < node.right.key:
            self.last_rotation_type = "RL"
            pivot = node.right.left
            self.last_pivot_key = pivot.key if pivot else None
            node.right = self._rotate_right(node.right)
            return self._rotate_left(node)

        return node

    def inorder(self):
        result = []
        self._inorder_rec(self.root, result)
        return result

    def _inorder_rec(self, node, result):
        if node:
            self._inorder_rec(node.left, result)
            result.append(node.key)
            self._inorder_rec(node.right, result)

    def get_node_bf(self, node):
        return self._balance_factor(node) if node else 0


# ---------------------------- GUI 应用程序（左右对比版）----------------------------
class AVLCompareApp:
    def __init__(self, master):
        self.master = master
        master.title("AVL树构建与旋转实操 - 左右对比")
        master.geometry("1200x700")

        self.insert_seq = [30, 20, 10, 25, 40, 35, 50]
        self.current_index = 0
        self.tree = AVLTree()          # 当前树
        self.prev_root = None          # 上一步的树根（深拷贝）

        # ---- 控制面板 ----
        control_frame = tk.Frame(master)
        control_frame.pack(pady=10)

        self.next_btn = tk.Button(control_frame, text="下一步 (插入)", command=self.next_step, width=15)
        self.next_btn.pack(side=tk.LEFT, padx=5)

        self.reset_btn = tk.Button(control_frame, text="重置", command=self.reset, width=10)
        self.reset_btn.pack(side=tk.LEFT, padx=5)

        self.auto_btn = tk.Button(control_frame, text="自动播放", command=self.auto_play, width=10)
        self.auto_btn.pack(side=tk.LEFT, padx=5)

        # ---- 信息显示 ----
        info_frame = tk.Frame(master)
        info_frame.pack(fill=tk.X, padx=10, pady=5)

        self.step_label = tk.Label(info_frame, text="当前步骤：未开始", font=("微软雅黑", 10))
        self.step_label.pack(anchor=tk.W)

        self.rotation_label = tk.Label(info_frame, text="旋转信息：无", font=("微软雅黑", 10), fg="blue")
        self.rotation_label.pack(anchor=tk.W)

        self.inorder_label = tk.Label(info_frame, text="当前树中序遍历：[]", font=("微软雅黑", 10), fg="green")
        self.inorder_label.pack(anchor=tk.W)

        # ---- 左右两棵树画布 ----
        canvas_frame = tk.Frame(master)
        canvas_frame.pack(pady=10, fill=tk.BOTH, expand=True)

        # 左侧画布（上一步）
        left_frame = tk.LabelFrame(canvas_frame, text="上一步（插入前）", font=("微软雅黑", 10, "bold"))
        left_frame.pack(side=tk.LEFT, fill=tk.BOTH, expand=True, padx=5)
        self.canvas_left = tk.Canvas(left_frame, bg="white", width=550, height=500)
        self.canvas_left.pack(fill=tk.BOTH, expand=True)

        # 右侧画布（当前步）
        right_frame = tk.LabelFrame(canvas_frame, text="当前步（插入后）", font=("微软雅黑", 10, "bold"))
        right_frame.pack(side=tk.RIGHT, fill=tk.BOTH, expand=True, padx=5)
        self.canvas_right = tk.Canvas(right_frame, bg="white", width=550, height=500)
        self.canvas_right.pack(fill=tk.BOTH, expand=True)

        # 初始绘制（空树）
        self.draw_both()

    # 重置所有状态
    def reset(self):
        self.tree = AVLTree()
        self.prev_root = None
        self.current_index = 0
        self.next_btn.config(state=tk.NORMAL)
        self.auto_btn.config(state=tk.NORMAL)
        self.update_info_display(None, None)
        self.draw_both()

    # 下一步：保存当前树 -> 插入 -> 更新信息 -> 绘制左右对比
    def next_step(self):
        if self.current_index >= len(self.insert_seq):
            messagebox.showinfo("完成", "所有数字均已插入完毕！")
            self.next_btn.config(state=tk.DISABLED)
            self.auto_btn.config(state=tk.DISABLED)
            return

        # 1. 保存上一步的树（深拷贝当前树的根）
        self.prev_root = AVLTree.copy_tree(self.tree.root)

        # 2. 插入新数
        key = self.insert_seq[self.current_index]
        rot_type, pivot = self.tree.insert(key)
        self.current_index += 1

        # 3. 更新显示信息
        self.update_info_display(rot_type, pivot)

        # 4. 同时绘制左右两棵树
        self.draw_both()

        # 如果已经完成全部插入，禁用下一步和自动按钮
        if self.current_index >= len(self.insert_seq):
            self.next_btn.config(state=tk.DISABLED)
            self.auto_btn.config(state=tk.DISABLED)

    # 自动播放
    def auto_play(self):
        if self.current_index >= len(self.insert_seq):
            messagebox.showinfo("完成", "所有数字均已插入完毕！")
            return
        self.auto_btn.config(state=tk.DISABLED)
        self._auto_step()

    def _auto_step(self):
        if self.current_index < len(self.insert_seq):
            self.next_step()
            self.master.after(1000, self._auto_step)
        else:
            self.auto_btn.config(state=tk.NORMAL)

    # 更新右侧文字信息
    def update_info_display(self, rot_type, pivot_key):
        step_info = f"已插入个数：{self.current_index}  "
        if self.current_index > 0:
            last_val = self.insert_seq[self.current_index-1]
            step_info += f"上一步插入：{last_val}"
        self.step_label.config(text=step_info)

        if rot_type:
            self.rotation_label.config(
                text=f"失衡类型：{rot_type}  |  旋转轴节点值：{pivot_key}",
                fg="red"
            )
        else:
            self.rotation_label.config(text="旋转信息：无失衡", fg="blue")

        # 显示当前树的中序遍历结果
        inorder_result = self.tree.inorder()
        self.inorder_label.config(text=f"当前树中序遍历：{inorder_result}")

    # 同时绘制左右两棵树
    def draw_both(self):
        # 清空两个画布
        self.canvas_left.delete("all")
        self.canvas_right.delete("all")

        # 绘制左侧（上一步的树）
        if self.prev_root is None:
            self.canvas_left.create_text(275, 250, text="空树（起始状态）", font=("微软雅黑", 14), fill="gray")
        else:
            # 计算合适的起始偏移量（根据树的宽度动态调整，这里简单使用固定偏移）
            start_x = 275   # 左侧画布中心 x 坐标
            start_y = 60
            start_offset = 100
            self._draw_tree_on_canvas(self.canvas_left, self.prev_root, start_x, start_y, start_offset, self.tree)

        # 绘制右侧（当前树）
        if self.tree.root is None:
            self.canvas_right.create_text(275, 250, text="空树", font=("微软雅黑", 14), fill="gray")
        else:
            start_x = 275
            start_y = 60
            start_offset = 100
            self._draw_tree_on_canvas(self.canvas_right, self.tree.root, start_x, start_y, start_offset, self.tree)

    # 静态绘制函数：在指定画布上绘制以 root 为根的树
    def _draw_tree_on_canvas(self, canvas, root, x, y, offset, tree_ref):
        """递归绘制树，tree_ref 是 AVLTree 实例（用于获取平衡因子）"""
        if not root:
            return

        # 辅助函数：计算子树宽度（节点个数）
        def subtree_width(node):
            if not node:
                return 0
            return 1 + subtree_width(node.left) + subtree_width(node.right)

        # 递归绘制节点
        def draw_node(node, cx, cy, x_offset):
            if not node:
                return

            # 左右子树宽度
            left_w = subtree_width(node.left)
            right_w = subtree_width(node.right)

            # 画连线
            if node.left:
                left_x = cx - x_offset
                left_y = cy + 70
                canvas.create_line(cx, cy, left_x, left_y, width=2, fill="gray")
                draw_node(node.left, left_x, left_y, x_offset * 0.6)
            if node.right:
                right_x = cx + x_offset
                right_y = cy + 70
                canvas.create_line(cx, cy, right_x, right_y, width=2, fill="gray")
                draw_node(node.right, right_x, right_y, x_offset * 0.6)

            # 画节点圆
            bf = tree_ref.get_node_bf(node)
            fill_color = "#FFCC99" if bf == 0 else "#FFB6C1"
            r = 20
            canvas.create_oval(cx-r, cy-r, cx+r, cy+r, fill=fill_color, outline="black", width=2)
            canvas.create_text(cx, cy, text=str(node.key), font=("Arial", 12, "bold"))
            canvas.create_text(cx, cy+r+12, text=f"bf={bf}", font=("Arial", 9), fill="darkgreen")

        draw_node(root, x, y, offset)


# ---------------------------- 主程序 ----------------------------
if __name__ == "__main__":
    root = tk.Tk()
    app = AVLCompareApp(root)
    root.mainloop()